# Code-Driven Pipeline (Inline)

[![Open in Colab](https://img.shields.io/badge/Open_in_Colab-F9AB00?style=for-the-badge&logo=googlecolab&logoColor=white)](https://colab.research.google.com/github/RealSlimMahdi/fleet-cdisc/blob/main/docs/examples/01_code_driven_pipeline.ipynb)

> **Interactive Playground** — Click the badge above to launch this notebook in Google Colab and experiment with Fleet live, no local setup needed.

This notebook demonstrates building clinical datasets entirely in Python code. It is best suited for quick ad-hoc analysis, smaller studies, or initial data discovery.

In [ ]:
try:
    import fleet  # noqa: F401 — registers .sdtm / .adam namespaces
except ModuleNotFoundError:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "fleet-cdisc@git+https://github.com/RealSlimMahdi/fleet-cdisc.git",
        ]
    )
    importlib.invalidate_caches()

import polars as pl

# Create some raw mock data
raw_dm = pl.from_repr("""\
┌─────────┬─────────┬─────┐
│ usubjid │ trt     │ age │
│ ---     │ ---     │ --- │
│ str     │ str     │ i64 │
╞═════════╪═════════╪═════╡
│ S001    │ Drug A  │ 35  │
│ S002    │ Placebo │ 42  │
└─────────┴─────────┴─────┘
""")

raw_vs = pl.from_repr("""\
┌─────────┬───────┬─────┐
│ usubjid │ test  │ val │
│ ---     │ ---   │ --- │
│ str     │ str   │ i64 │
╞═════════╪═══════╪═════╡
│ S001    │ SysBP │ 120 │
│ S001    │ DiaBP │ 80  │
│ S002    │ SysBP │ 130 │
└─────────┴───────┴─────┘
""")

## 1. Derive SDTM

We use the `.sdtm` namespace to standardize the raw inputs.

In [2]:
dm = (
    raw_dm.sdtm.assign("DOMAIN", value="DM")
    .assign("USUBJID", source="usubjid")
    .assign("ARM", source="trt")
    .assign("AGE", source="age")
    .to_frame()
)

vs = (
    raw_vs.sdtm.assign("DOMAIN", value="VS")
    .assign("USUBJID", source="usubjid")
    .assign("VSTESTCD", source="test")
    .assign("VSSTRESN", source="val")
    .to_frame()
)

print(dm)
print(vs)

shape: (2, 7)
┌─────────┬─────────┬─────┬────────┬─────────┬─────────┬─────┐
│ usubjid ┆ trt     ┆ age ┆ DOMAIN ┆ USUBJID ┆ ARM     ┆ AGE │
│ ---     ┆ ---     ┆ --- ┆ ---    ┆ ---     ┆ ---     ┆ --- │
│ str     ┆ str     ┆ i64 ┆ str    ┆ str     ┆ str     ┆ i64 │
╞═════════╪═════════╪═════╪════════╪═════════╪═════════╪═════╡
│ S001    ┆ Drug A  ┆ 35  ┆ DM     ┆ S001    ┆ Drug A  ┆ 35  │
│ S002    ┆ Placebo ┆ 42  ┆ DM     ┆ S002    ┆ Placebo ┆ 42  │
└─────────┴─────────┴─────┴────────┴─────────┴─────────┴─────┘
shape: (3, 7)
┌─────────┬───────┬─────┬────────┬─────────┬──────────┬──────────┐
│ usubjid ┆ test  ┆ val ┆ DOMAIN ┆ USUBJID ┆ VSTESTCD ┆ VSSTRESN │
│ ---     ┆ ---   ┆ --- ┆ ---    ┆ ---     ┆ ---      ┆ ---      │
│ str     ┆ str   ┆ i64 ┆ str    ┆ str     ┆ str      ┆ i64      │
╞═════════╪═══════╪═════╪════════╪═════════╪══════════╪══════════╡
│ S001    ┆ SysBP ┆ 120 ┆ VS     ┆ S001    ┆ SysBP    ┆ 120      │
│ S001    ┆ DiaBP ┆ 80  ┆ VS     ┆ S001    ┆ DiaBP    ┆ 80       │

## 2. Derive ADaM

We use the `.adam` namespace. Notice how easily we define a custom derivation to flag subjects who are older than 40.

In [3]:
adsl = (
    dm.adam.adsl.source("USUBJID")
    .source("ARM")
    .source("AGE")
    # Custom inline derivation not covered by standard library verbs
    .derivation(
        "AGE40FL", expression=pl.when(pl.col("AGE") > 40).then(pl.lit("Y")).otherwise(pl.lit("N"))
    )
    .to_frame()
)

print(adsl)

shape: (2, 8)
┌─────────┬─────────┬─────┬────────┬─────────┬─────────┬─────┬─────────┐
│ usubjid ┆ trt     ┆ age ┆ DOMAIN ┆ USUBJID ┆ ARM     ┆ AGE ┆ AGE40FL │
│ ---     ┆ ---     ┆ --- ┆ ---    ┆ ---     ┆ ---     ┆ --- ┆ ---     │
│ str     ┆ str     ┆ i64 ┆ str    ┆ str     ┆ str     ┆ i64 ┆ str     │
╞═════════╪═════════╪═════╪════════╪═════════╪═════════╪═════╪═════════╡
│ S001    ┆ Drug A  ┆ 35  ┆ DM     ┆ S001    ┆ Drug A  ┆ 35  ┆ N       │
│ S002    ┆ Placebo ┆ 42  ┆ DM     ┆ S002    ┆ Placebo ┆ 42  ┆ Y       │
└─────────┴─────────┴─────┴────────┴─────────┴─────────┴─────┴─────────┘
